# 01_流式输出

**来源:** [LangChain Docs — Streaming](https://docs.langchain.com/oss/python/deepagents/streaming)

本 Notebook 演示 Deep Agents 的流式输出能力。Deep Agents 基于 LangGraph 的流式基础设施构建，支持从主 Agent 和子 Agent 中实时流式输出更新、LLM Token、工具调用和自定义事件。

## 1. 环境准备与导入

In [ ]:
# 安装依赖
# pip install deepagents langchain langgraph

from deepagents import create_deep_agent
from langgraph.checkpoint.memory import MemorySaver
from langgraph.config import get_stream_writer
from langchain.tools import tool
import time

## 2. 启用子图流（Subgraph Streaming）

Deep Agents 使用 LangGraph 的子图流式功能来暴露子 Agent 的执行事件。启用 `subgraphs=True` 即可接收子 Agent 事件。

In [ ]:
# 创建带有子 Agent 的深度 Agent
agent = create_deep_agent(
    model="google_genai:gemini-3.5-flash",
    system_prompt="你是一个有用的研究助手",
    subagents=[
        {
            "name": "researcher",
            "description": "深入调研某个主题",
            "system_prompt": "你是一个细致的研究员。",
        },
    ],
)

# 使用 subgraphs=True 启用子图流式输出
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "研究一下量子计算的最新进展"}]},
    stream_mode="updates",
    subgraphs=True,  # 启用子图流式
    version="v2",    # 使用 v2 流式格式
):
    if chunk["type"] == "updates":
        if chunk["ns"]:
            # 子 Agent 事件 - namespace 标识来源
            print(f"[子 Agent: {chunk['ns']}]")
        else:
            # 主 Agent 事件
            print("[主 Agent]")
        print(chunk["data"])

## 3. Namespace 路由

当启用子图流式后，每个流式事件包含一个 `namespace`（命名空间），用于标识产生该事件的 Agent。Namespace 是节点名称和任务 ID 的路径。

| Namespace | 来源 |
|-----------|------|
| `()` (空) | 主 Agent |
| `("tools:abc123",)` | 主 Agent 的 task 工具调用产生的子 Agent |
| `("tools:abc123", "model_request:def456")` | 子 Agent 内部的模型请求节点 |

In [ ]:
# 使用 Namespace 将事件路由到正确的 UI 组件
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "帮我规划假期"}]},
    stream_mode="updates",
    subgraphs=True,
    version="v2",
):
    if chunk["type"] == "updates":
        # 检查事件是否来自子 Agent
        is_subagent = any(
            segment.startswith("tools:") for segment in chunk["ns"]
        )

        if is_subagent:
            # 从 namespace 中提取工具调用 ID
            tool_call_id = next(
                s.split(":")[1] for s in chunk["ns"] if s.startswith("tools:")
            )
            print(f"子 Agent {tool_call_id}: {chunk['data']}")
        else:
            print(f"主 Agent: {chunk['data']}")

## 4. 追踪子 Agent 进度

使用 `stream_mode="updates"` 追踪子 Agent 的进度，查看哪些子 Agent 处于活动状态以及它们完成了什么工作。

In [ ]:
# 创建带子 Agent 的协调 Agent
agent = create_deep_agent(
    model="google_genai:gemini-3.5-flash",
    system_prompt=(
        "你是一个项目协调员。始终将研究任务委托给你的研究员子 Agent。"
        "最终回答保持一句话。"
    ),
    subagents=[
        {
            "name": "researcher",
            "description": "全面调研各类主题",
            "system_prompt": (
                "你是一个细致的研究员。研究给定主题并用 2-3 句话提供简洁摘要。"
            ),
        },
    ],
)

# 追踪子 Agent 执行进度
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "写一篇关于 AI 安全的简短摘要"}]},
    stream_mode="updates",
    subgraphs=True,
    version="v2",
):
    if chunk["type"] == "updates":
        # 主 Agent 更新（空 namespace）
        if not chunk["ns"]:
            for node_name, data in chunk["data"].items():
                if node_name == "tools":
                    # 子 Agent 结果返回给主 Agent
                    for msg in data.get("messages", []):
                        if msg.type == "tool":
                            print(f"\n子 Agent 完成: {msg.name}")
                            print(f"  结果: {str(msg.content)[:200]}...")
                else:
                    print(f"[主 Agent] 步骤: {node_name}")

        # 子 Agent 更新（非空 namespace）
        else:
            for node_name, data in chunk["data"].items():
                print(f"  [{chunk['ns'][0]}] 步骤: {node_name}")

## 5. 流式输出 LLM Token

使用 `stream_mode="messages"` 从主 Agent 和子 Agent 流式输出独立的 Token。每条消息事件包含标识来源 Agent 的元数据。

In [ ]:
# 流式输出 LLM Token
current_source = ""

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "研究量子计算进展"}]},
    stream_mode="messages",
    subgraphs=True,
    version="v2",
):
    if chunk["type"] == "messages":
        token, metadata = chunk["data"]

        # 检查事件是否来自子 Agent（namespace 包含 "tools:"）
        is_subagent = any(s.startswith("tools:") for s in chunk["ns"])

        if is_subagent:
            # 来自子 Agent 的 Token
            subagent_ns = next(s for s in chunk["ns"] if s.startswith("tools:"))
            if subagent_ns != current_source:
                print(f"\n\n--- [子 Agent: {subagent_ns}] ---")
                current_source = subagent_ns
            if token.content:
                print(token.content, end="", flush=True)
        else:
            # 来自主 Agent 的 Token
            if "main" != current_source:
                print("\n\n--- [主 Agent] ---")
                current_source = "main"
            if token.content:
                print(token.content, end="", flush=True)

print()

## 6. 流式输出工具调用

当子 Agent 使用工具时，可以流式输出工具调用事件，展示每个子 Agent 正在做什么。

In [ ]:
# 流式输出工具调用事件
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "研究最近的量子计算进展"}]},
    stream_mode="messages",
    subgraphs=True,
    version="v2",
):
    if chunk["type"] == "messages":
        token, metadata = chunk["data"]

        # 标识来源："main" 或子 Agent namespace 段
        is_subagent = any(s.startswith("tools:") for s in chunk["ns"])
        source = next((s for s in chunk["ns"] if s.startswith("tools:")), "main") if is_subagent else "main"

        # 工具调用块（流式工具调用）
        if token.tool_call_chunks:
            for tc in token.tool_call_chunks:
                if tc.get("name"):
                    print(f"\n[{source}] 工具调用: {tc['name']}")
                # 参数流式输出 - 增量写入
                if tc.get("args"):
                    print(tc["args"], end="", flush=True)

        # 工具结果
        if token.type == "tool":
            print(f"\n[{source}] 工具结果 [{token.name}]: {str(token.content)[:150]}")

        # 常规 AI 内容（跳过工具调用消息）
        if token.type == "ai" and token.content and not token.tool_call_chunks:
            print(token.content, end="", flush=True)

print()

## 7. 自定义进度更新

在子 Agent 的工具中使用 `get_stream_writer` 发送自定义进度事件。

In [ ]:
# 自定义工具：模拟数据分析并发送进度更新
@tool
def analyze_data(topic: str) -> str:
    """对给定主题进行数据分析。

    此工具执行实际分析并发出进度更新。
    对于任何分析请求，你必须调用此工具。
    """
    writer = get_stream_writer()

    # 发送进度阶段
    writer({"status": "starting", "topic": topic, "progress": 0})
    time.sleep(0.5)

    writer({"status": "analyzing", "progress": 50})
    time.sleep(0.5)

    writer({"status": "complete", "progress": 100})
    return (
        f'对 "{topic}" 的分析结果：客户满意度 85% 正面，'
        "主要由产品质量和支持响应时间驱动。"
    )


# 创建带自定义工具的 Agent
agent = create_deep_agent(
    model="google_genai:gemini-3.5-flash",
    system_prompt=(
        "你是一个协调员。对于任何分析请求，你必须委托给分析师子 Agent。"
        "收到结果后用一句话总结。"
    ),
    subagents=[
        {
            "name": "analyst",
            "description": "执行数据分析并实时跟踪进度",
            "system_prompt": (
                "你是一个数据分析师。你必须为每个分析请求调用 analyze_data 工具。"
                "分析完成后，报告结果。"
            ),
            "tools": [analyze_data],
        },
    ],
)

# 使用 custom mode 接收自定义事件
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "分析客户满意度趋势"}]},
    stream_mode="custom",
    subgraphs=True,
    version="v2",
):
    if chunk["type"] == "custom":
        is_subagent = any(s.startswith("tools:") for s in chunk["ns"])
        if is_subagent:
            subagent_ns = next(s for s in chunk["ns"] if s.startswith("tools:"))
            print(f"[{subagent_ns}] {chunk['data']}")
        else:
            print(f"[主] {chunk['data']}")

## 8. 组合多种流式模式

组合多种流式模式以获得 Agent 执行的完整视图。

In [ ]:
# 只显示有意义的节点名称，跳过内部中间件步骤
INTERESTING_NODES = {"model_request", "tools"}

last_source = ""
mid_line = False  # 当写入 Token 但没有换行时为 True

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "分析远程工作对团队生产力的影响"}]},
    stream_mode=["updates", "messages", "custom"],
    subgraphs=True,
    version="v2",
):
    is_subagent = any(s.startswith("tools:") for s in chunk["ns"])
    source = "子 Agent" if is_subagent else "主"

    if chunk["type"] == "updates":
        for node_name in chunk["data"]:
            if node_name not in INTERESTING_NODES:
                continue
            if mid_line:
                print()
                mid_line = False
            print(f"[{source}] 步骤: {node_name}")

    elif chunk["type"] == "messages":
        token, metadata = chunk["data"]
        if token.content:
            # 来源变化时打印头部
            if source != last_source:
                if mid_line:
                    print()
                    mid_line = False
                print(f"\n[{source}] ", end="")
                last_source = source
            print(token.content, end="", flush=True)
            mid_line = True

    elif chunk["type"] == "custom":
        if mid_line:
            print()
            mid_line = False
        print(f"[{source}] 自定义事件: {chunk['data']}")

print()

## 9. 追踪子 Agent 生命周期

监控子 Agent 何时启动、运行和完成。

In [ ]:
# 追踪子 Agent 的完整生命周期
active_subagents = {}

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "研究最新的 AI 安全进展"}]},
    stream_mode="updates",
    subgraphs=True,
    version="v2",
):
    if chunk["type"] == "updates":
        for node_name, data in chunk["data"].items():
            # 阶段 1: 检测子 Agent 启动
            if not chunk["ns"] and node_name == "model_request":
                for msg in data.get("messages", []):
                    for tc in getattr(msg, "tool_calls", []):
                        if tc["name"] == "task":
                            active_subagents[tc["id"]] = {
                                "type": tc["args"].get("subagent_type"),
                                "description": tc["args"].get("description", "")[:80],
                                "status": "pending",
                            }
                            print(
                                f'[生命周期] 待定 → 子 Agent "{tc["args"].get("subagent_type")}" '
                                f'({tc["id"]})'
                            )

            # 阶段 2: 检测子 Agent 运行中
            if chunk["ns"] and chunk["ns"][0].startswith("tools:"):
                pregel_id = chunk["ns"][0].split(":")[1]
                for sub_id, sub in active_subagents.items():
                    if sub["status"] == "pending":
                        sub["status"] = "running"
                        print(
                            f'[生命周期] 运行中 → 子 Agent "{sub["type"]}" '
                            f"(pregel: {pregel_id})"
                        )
                        break

            # 阶段 3: 检测子 Agent 完成
            if not chunk["ns"] and node_name == "tools":
                for msg in data.get("messages", []):
                    if msg.type == "tool":
                        sub = active_subagents.get(msg.tool_call_id)
                        if sub:
                            sub["status"] = "complete"
                            print(
                                f'[生命周期] 完成 → 子 Agent "{sub["type"]}" '
                                f"({msg.tool_call_id})"
                            )
                            print(f"  结果预览: {str(msg.content)[:120]}...")

# 打印最终状态
print("\n--- 最终子 Agent 状态 ---")
for sub_id, sub in active_subagents.items():
    print(f"  {sub['type']}: {sub['status']}")

## 10. v2 流式格式

所有示例使用 `version="v2"` 格式（需要 LangGraph >= 1.1）。每个块是包含 `type`、`ns` 和 `data` 键的 `StreamPart` 字典——无论流式模式、模式数量或子图设置如何，形状保持一致。

In [ ]:
# v2 格式：统一格式，无需嵌套元组解包
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "研究量子计算"}]},
    stream_mode=["updates", "messages", "custom"],
    subgraphs=True,
    version="v2",
):
    print(chunk["type"])  # "updates", "messages", 或 "custom"
    print(chunk["ns"])    # () 为主 Agent, ("tools:<id>",) 为子 Agent
    print(chunk["data"])  # 负载

---

**参考链接**
- [Deep Agents — Streaming](https://docs.langchain.com/oss/python/deepagents/streaming)
- [LangGraph 流式文档](https://langchain-ai.github.io/langgraph/)
- [子 Agent](https://docs.langchain.com/oss/python/deepagents/subagents)
- [前端流式输出](https://docs.langchain.com/oss/python/deepagents/frontend-streaming)
- [LangChain 事件流式](https://docs.langchain.com/oss/python/deepagents/event-streaming)